![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp-workshop/blob/master/tutorials/vlm-workshop/1.visual_deid.ipynb)

# 🔒 Visual De-Identification

> A single missed PHI entity in a released document is a HIPAA violation. Manual redaction takes 5-15 minutes per page and still misses things.

**What you'll see:**
- OCR that returns every text line with its bounding box coordinates — not just text, but *where* on the page
- Surgical redaction of individual PHI fields (names, SSNs, dates) — not black bars over entire paragraphs
- **100% PHI recall** across 30 clinical documents — zero missed entities
- The same pipeline reading handwritten prescriptions and extracting drug names, dosages, and routes


| Segment | Dataset | Docs | What Happens |
|---------|---------|:----:|--------------|
| Digital medical documents | [JohnSnowLabs/pdf-deid-dataset](https://github.com/JohnSnowLabs/pdf-deid-dataset) | 30 | OCR → DEID → redaction, benchmarked against PHI ground truth |
| Handwritten prescriptions | Synthetic handwritten Rx (local) | 20 | OCR → DEID → posology NER (drug/dosage/route extraction) |

**Models**: jsl_vision_ocr_parsing_1_0 (OCR + bounding boxes) · Spark NLP for Healthcare (DEID + NER)
**Evaluation**: PHI recall/precision vs ground truth, region accuracy

**Bottom line**: 1,160 docs/hr at 100% recall — saves ~`$150-200K/yr` at 200 docs/day, and one missed PHI entity can cost `$50K+` in HIPAA fines.

In [ ]:
# -- Colab Setup ---------------------------------------------------------------
# Run this cell once to install dependencies and download data.
# Skipped automatically when running locally.
import os, sys
if "COLAB_RELEASE_TAG" in os.environ:
    # 1) Python dependencies
    !pip install -q pandas numpy tqdm pillow datasets matplotlib seaborn scikit-learn openai python-dotenv faiss-cpu requests PyMuPDF

    # 2) Source files (utils/, config)
    !wget -qO git_downloads.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/git_downloads.tar.gz"
    !tar xzf git_downloads.tar.gz && rm git_downloads.tar.gz

    # 3) Cached predictions + datasets
    !wget -qO nb1_data.tar.gz "https://ckl-emr-bucket.s3.us-east-1.amazonaws.com/vlm-workshop/nb1_data.tar.gz"
    !tar xzf nb1_data.tar.gz && rm nb1_data.tar.gz

    print("Setup complete — all dependencies and data downloaded.")


In [ ]:
# --- Imports & Configuration -----------------------------------------------
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')

from utils.config import OCR_BASE_URL, OCR_MODEL, SPARKNLP_BASE_URL, OUTPUT_DIR
from utils import *
from utils.ocr_deid import (
    load_digital_docs, load_handwritten_docs,
    preview_digital_docs,
    run_ocr_deid_cached, export_ocr_deid,
    display_ocr_deid_results,

    show_phi_accuracy, enrich_posology,
)

NB_NAME = "nb1_visual_deid"
set_cache_model(OCR_MODEL)
USE_CACHE = True
USE_VLLM_SERVER = not USE_CACHE
VLLM_BASE_URL = OCR_BASE_URL
VLLM_MODEL = OCR_MODEL
DIGITAL_N = 30
HANDWRITTEN_N = 20

OUTPUT_DIR = OUTPUT_DIR / "ocr_bb_deid"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Server Health Check ---------------------------------------------------
if USE_VLLM_SERVER:
    check_server(VLLM_BASE_URL, VLLM_MODEL)
    check_sparknlp(SPARKNLP_BASE_URL)

# --- Use Case --------------------------------------------------------------
show_use_case('visual_deid')

---
## 📄 Section 1: Digital Medical Documents

**Dataset**: [JohnSnowLabs/pdf-deid-dataset](https://github.com/JohnSnowLabs/pdf-deid-dataset) — 120 synthetic medical PDFs across 3 difficulty levels (Easy/Medium/Hard), each containing ~41-52 PHI entities (names, DOB, SSN, hospital IDs, phone numbers, addresses). Sampling 30 (10 per level).

Ground truth PHI annotations used for precision/recall benchmarking.

```mermaid
flowchart LR
    A["🖼️ Scanned / Digital Document"] --> B["jsl_vision_ocr_parsing_1_0<br>(OCR + bounding boxes)"]
    B --> C["Spark NLP DEID<br>(PHI detection)"]
    C --> D["✅ Redacted PDF<br>(precise redaction)"]
    style B fill:#6366f1,color:#fff,stroke:#4f46e5
    style C fill:#14b8a6,color:#fff,stroke:#0d9488
    style D fill:#10b981,color:#fff,stroke:#059669
```


In [ ]:
digital_images, digital_ids, digital_levels, digital_phi, _ = load_digital_docs(DIGITAL_N)
preview_digital_docs(digital_images, digital_ids, digital_levels, digital_phi)

In [ ]:
digital_results = run_ocr_deid_cached(
    NB_NAME, "digital_results", USE_CACHE, digital_images, digital_ids,
    VLLM_BASE_URL, VLLM_MODEL, gt_phi_list=digital_phi, sparknlp_url=SPARKNLP_BASE_URL, desc="Digital OCR+DEID",
)

In [ ]:
digital_meta = [{"Difficulty": lvl} for lvl in digital_levels]
display_ocr_deid_results(digital_images, digital_results, title="Digital — OCR + DEID", metadata=digital_meta)

In [ ]:
show_phi_accuracy(digital_results)

---
## Cloud VLM Comparison: Claude Sonnet 4.6 & GPT-5.4

**Can frontier cloud VLMs match a specialized on-prem pipeline for HIPAA de-identification?**

We send the same 30 digital medical PDFs to **Claude Sonnet 4.6** (Anthropic) and **GPT-5.4** (OpenAI) via OpenRouter, asking each model to identify all PHI entities in a single shot — no OCR step, no NER pipeline, just raw vision + reasoning.

Key differences:
- **Our pipeline**: JSL Vision OCR (line-level bounding boxes) + Spark NLP DEID (clinical NER) → precise redaction
- **Cloud VLMs**: One-shot entity extraction → text-only PHI list (no spatial coordinates for redaction)

In [ ]:
from utils.ocr_deid import run_cloud_deid, show_cloud_comparison
from utils.config import OPENROUTER_BASE_URL, OPENROUTER_API_KEY

CLOUD_MODELS = {
    "Claude Sonnet 4.6": "anthropic/claude-sonnet-4.6",
    "GPT-5.4":           "openai/gpt-5.4",
}

cloud_results = {}
for label, model_id in CLOUD_MODELS.items():
    cache_key = f"cloud_deid_{model_id.replace('/', '_')}"
    if USE_CACHE and has_nb_cache(NB_NAME, cache_key):
        cached = load_nb_cache(NB_NAME, cache_key)
        cloud_results[label] = cached["results"]
        n = len(cached["results"])
        ents = sum(len(r.get("phi_entities", [])) for r in cached["results"])
        print(f"📦 {label}: loaded {n} docs from cache ({ents} entities)")
    else:
        cloud_results[label] = run_cloud_deid(
            digital_images, digital_ids, model_id, OPENROUTER_API_KEY,
            base_url=OPENROUTER_BASE_URL, gt_phi_list=digital_phi,
            desc=f"Cloud DEID ({label})",
        )
        save_nb_cache(NB_NAME, cache_key, {"results": cloud_results[label]})

In [ ]:
show_cloud_comparison(digital_results, cloud_results)

In [ ]:
from utils.ocr_deid import plot_cloud_comparison
plot_cloud_comparison(digital_results, cloud_results)

---
## ✍️ Section 2: Handwritten Prescriptions

**Dataset**: 129 handwritten medical prescriptions from `dataset/handwriting-ocr-data/`.
Blacklisted unreadable images. Tiny images (min dimension < 800px) auto-upscaled with LANCZOS.

DEID uses Spark NLP clinical de-identification (when available) for real PHI detection.

In [ ]:
handwritten_images, handwritten_ids = load_handwritten_docs(
    "./dataset/handwriting-ocr-data", HANDWRITTEN_N,
)

handwritten_preview = [{"doc_id": did, "source": "handwriting-ocr-data"} for did in handwritten_ids]
display_document_analysis(handwritten_images, handwritten_preview, title="Handwritten Prescriptions — Before OCR")

In [ ]:
handwritten_results = run_ocr_deid_cached(
    NB_NAME, "handwritten_results", USE_CACHE, handwritten_images, handwritten_ids,
    VLLM_BASE_URL, VLLM_MODEL, sparknlp_url=SPARKNLP_BASE_URL, desc="Handwritten OCR+DEID",
)

In [ ]:
display_ocr_deid_results(handwritten_images, handwritten_results, title="Handwritten — OCR + DEID")

---
## Impact & ROI

### Accuracy (Digital Medical Documents — 30 PDFs)

| Metric | JSL Vision OCR + Spark NLP | Claude Sonnet 4.6 | GPT-5.4 |
|--------|:---------------------:|:---------------:|:-------:|
| **PHI Recall** | **100%** (524/524) | 63.0% | 65.1% |
| **PHI Precision** | 90.1% | 95.5% | 94.3% |
| **Region Accuracy** | 94.1% | — | — |
| **Bounding Boxes** | Yes | No | No |
| **PHI Leaves Network** | No | Yes | Yes |

100% recall means **zero missed PHI** across all 30 documents (Easy/Medium/Hard). In HIPAA compliance, a missed entity is a violation — recall is the metric that matters. Claude and GPT-5.4 miss ~35% of PHI entities despite high precision.

### Speed & Cost

| Metric | Manual Redaction | JSL (On-Prem) | Claude Sonnet 4.6 | GPT-5.4 |
|--------|:---------------:|:---------------------:|:---------------:|:-------:|
| Time per document | 5–15 min | **3.1 sec** | 11.1 sec | 4.9 sec |
| Batch of 30 docs | 2.5–7.5 hours | **93 sec** | 334 sec | 147 sec |
| Throughput | ~4–12 docs/hr | **1,160 docs/hr** | ~324 docs/hr | ~735 docs/hr |
| API cost per doc | — | `$0` (on-prem) | ~`$0.03` | ~`$0.02` |

### Annual Cost Savings (JSL On-Prem vs Alternatives)

*Assumptions: HIPAA compliance officer at `$25/hr`, 250 working days/yr.*

| Scale | Manual Cost | JSL On-Prem | Cloud VLM (API) | JSL Savings vs Manual |
|-------|:----------:|:-----------:|:---------------:|:---------------------:|
| 50 docs/day | `$52K/yr` | ~`$5K/yr` | ~`$10–19K/yr` + HIPAA risk | **~`$47K/yr`** |
| 200 docs/day | `$156–208K/yr` | ~`$8K/yr` | ~`$40–75K/yr` + HIPAA risk | **~`$150–200K/yr`** |
| 1,000 docs/day | `$780K/yr` | ~`$15K/yr` | ~`$200–375K/yr` + HIPAA risk | **~`$765K/yr`** |

**HIPAA violation avoidance**: A single PHI breach costs `$100–$50K` per violation (up to `$1.5M/yr` per category). 100% recall eliminates the #1 source of accidental disclosure — missed entities in released documents. Cloud VLMs at 65% recall would miss ~175 PHI entities across these 30 docs alone.

### Why This Matters

- **HIPAA compliance at scale**: 100% PHI recall vs 65% for frontier cloud VLMs — eliminates the #1 risk
- **Bounding-box precision**: Surgical redaction of individual fields — cloud VLMs cannot localize text on the page
- **On-prem deployment**: PHI never leaves your network — cloud APIs require sending patient data to third parties
- **Handwritten support**: Same pipeline handles handwritten prescriptions — the hardest OCR task
- **Drug extraction**: Posology NER on handwritten Rx extracts drug names, dosages, frequencies linked to RxNorm codes
- **3.5x faster, 10x cheaper**: 3.1s/doc on-prem vs 4.9–11.1s/doc cloud, zero per-document API cost

---
## 💾 Export

In [ ]:
export_ocr_deid({"digital": digital_results, "handwritten": handwritten_results}, OUTPUT_DIR / "ocr_bbox_results.json")